# A few words about the Kaggle's Python Environment

The python environment in Kaggle already comes with many helpful analytics libraries:
- Numpy
- Pandas
- Matplotlib
- Seaborn
- SKLearn
- ELI5

The environment is defined by the `kaggle/python` Docker image. If you want to learn more on what is the configuration of the cloud image you are running your code, please check the [GitHub repository](https://github.com/kaggle/docker-python).

## Input

There is only one file available for reading in this notebook, the **German Credit dataset** we will use in our hands-on session. 
The dataset is available at `/kaggle/input/german-credit-data-with-risk/german_credit_data.csv`.

## Saving data

You can write up to **20 GB** to the current directory`kaggle/working` that gets preserved across sessions. That is, next time you open this notebook and start the session, the file will be available to you.

You can also write temporary files to `/kaggle/temp/`, but they won't be saved outside of the current session

## Helpful shortcuts

All the shortcuts below work when the target cell is selected (not when editing). 
- a: To insert cell above
- b: To insert cell below   
- dd: To delete cell
- m: Change the cell to markdown 
- y: Change the cell to code

For more shortcuts, please check this [notebook](https://www.kaggle.com/naushads/keyboard-shortcuts-for-kaggle-kernels).


In [ ]:
# Checking the files available in this notebook 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Context: German Credit Report

The original dataset contains entries with 20 categorial/symbolic attributes prepared by Prof. Hofmann. In this dataset, each entry represents a person who takes a credit by a bank. Each person is classified as good or bad credit risks according to the set of attributes. The original dataset can be found [here](https://archive.ics.uci.edu/ml/datasets/Statlog+%28German+Credit+Data%29)

We will use a simplified version of the dataset with only **9 features** in our class. 

In [ ]:
### Importing the general libraries we shall use in this notebook
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns # plotting library that use matplot in background
import matplotlib.pyplot as plt # to plot some parameters in seaborn

# Reading the dataset
dataset = pd.read_csv('../input/german-credit-data-with-risk/german_credit_data.csv', index_col=None)

In [ ]:
dataset.head()

In [ ]:
# Cleaning up the dataset
dataset = dataset.drop('Unnamed: 0', axis='columns')
dataset.head(n=20)

## Part 1: Exploring the dataset

1. Explore the dataset features and the target variable
 - How much data do we have?
 - Do we have any missing data? (NaN)
 - What are the types of features in our dataset?
 - How is the distribution of the target variable?

Tip: Check out the [Pandas API](https://pandas.pydata.org/) for functions that can help with data exploration.
 

In [ ]:
# How much data do we have?


# Do we have any missing data?


# What are the types of features (numerical vs categorical) ?


# What is the distribution of the target variable? 



Write your observations of Part 1 here
- Our dataset has X rows and Y columns
- [...]


# Part 2: Analysing the distribution and relationship of features

In this part of the hands-on session, we shall explore the relationship between features and their respective distribution in the dataset.
- Do we have a biased dataset?
- How some features relate to good/bad credit?



In [ ]:
sns.set_context('talk', font_scale=.9)
# Example of types of analysis that can be done

# Count plot helps us visualize the number of elements per category
sns.countplot(data=dataset, x='Saving accounts', hue='Risk')
plt.show()

# Box plot helps us see the mean value of a category "Sex" per "Age" in our dataset
sns.catplot(data=dataset, x='Sex', y='Age', kind='box')
plt.show()

# Split violin plots help us contrast the distribution across a hue value "Risk"
sns.violinplot(data=dataset, x='Sex', y='Age', hue='Risk', split=True)
plt.show()

# Displot help us visualize the distribution with histograms
sns.displot(data=dataset, row='Sex', y='Purpose', col='Risk')
plt.show()

# Experiment by analysing other features in the dataset!
- What about Checking account vs Age vs Risk?
- What about Savings account vs Credit amount?
- What people from different type of Checking Account (little, moderate, rich) ask credit for? 
- ...


In [ ]:
# Reuse the plotting functions above and explore different relationships between features of your dataset



Write some observations you found on the distribution across features here
- Observation 1...

# Part 3. Preparing the dataset for our Modeling (Feature Engineering) (Together)

Now that we have explored some of the features our dataset contains, we should evaluate their quality and the potential for extracting more informative features by applying some domain knowledge, or combining features together. 

- Are there features that could be better represented?
- Can we extract other features based on the current set?

In [ ]:
dataset.head()

In [ ]:
# Could age be better represented?

# Let's see how age is distributed
sns.displot(dataset['Age'])
plt.show()

In [ ]:
#Let us split age into categories
interval = (18, 25, 35, 60, 120)
cats = ['Young Adult', 'Adult', 'Senior', 'Elder']
dataset["Age_cat"] = pd.cut(dataset['Age'], interval, labels=cats)

In [ ]:
dataset.head()

In [ ]:
sns.set(rc={'figure.figsize':(11.7,16)})

# Could Credit Amount be better represented?
# Let's see how is credit amount distributed
sns.displot(dataset['Credit amount'])
plt.show()

In [ ]:
# Another way to represent long tail numerical distributions is to transform them
# using e.g., a log function
sns.displot(np.log10(dataset['Credit amount']))
plt.show()

# Apply the new distribution to the dataset
dataset['Credit amount'] = np.log10(dataset['Credit amount'])

In [ ]:
dataset.head()

In [ ]:
# Dealing with Missing values of Saving account and Checking account
dataset['Saving accounts'] = dataset['Saving accounts'].fillna('no_inf')
dataset['Checking account'] = dataset['Checking account'].fillna('no_inf')

In [ ]:
dataset.head()

### Encoding categorical features

Categorical features poses a problem to (some) ML models: 
- How to represent categorical features like Housing = {own, free}?

**Solution 1 - Integer encoding:** We could represent them using numerical values - Housing = {own = 1, free = 2} 

| Housing | 
| -- |
| 1 | 
| 2 | 

- The problem of this representation is that it assumes an ordered relationship: 
 - does "own" housing comes before than "free" housing? Does this question even makes sense? 

**Solution 2 - One-Hot encoding:** Instead of represengin Housing = {own = 1, free = 2} we turn Housing into a matrix:


| Housing_own | Housing_free | 
| -- | -- |
| 1 | 0 | 
| 0 | 1 | 


In [ ]:
# Sci-kit learn has a proper OneHotEncoder which we could use as part of our pipeline. However, 
# for exploration reasons, the OneHotEncoder of Sci-kit learner does not keep track of what value

def one_hot_encoder(df, column_name, exclude_col = False):
    merged_df = df.merge(pd.get_dummies(df[column_name], drop_first=False, prefix=column_name), left_index=True, right_index=True)
    if exclude_col:
        del merged_df[column_name] # Exclude the original column
    return merged_df

In [ ]:
# Let us see our columns before we apply the one-hot-encoding
dataset.columns

In [ ]:
# Given the change is more meaningful, let us copy to another dataset (we can always compare them both at a later time)
dataset_ready = dataset.copy()

# Categorical features to encode using One-Hot encoding
category_features = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose', 'Risk', 'Age_cat']

for cat in category_features:
    dataset_ready = one_hot_encoder(dataset_ready, cat, exclude_col=True)

In [ ]:
# Let us inspect how the categorical features have not been expanded in each column 
dataset_ready.columns

In [ ]:
# Let us inspect how the categorical features have not been expanded in each column 
dataset_ready.columns

dataset_ready.head()

# Part 4. Predicting the Risk (Modeling)

First, we will split the dataset into:
- Features (X) and target variable (y)
- Training (75%) and test sets (25%)

In [ ]:
# Importing the libraries we will use in this part of the class
from sklearn.model_selection import train_test_split, KFold, cross_val_score # to split the data
from sklearn.metrics import accuracy_score, plot_confusion_matrix, classification_report, f1_score, precision_score, recall_score #To evaluate our model

from sklearn.model_selection import GridSearchCV

# Algorithmns models to be compared
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
# TODO: Add here any new model you may want to try out (ANN, etc.)

# Creating the X and y variables
dataset_ready_x = dataset_ready.drop(['Risk_bad', 'Risk_good', 'Age', 'Sex_male'], axis='columns')
X = dataset_ready_x.values
feature_names = dataset_ready_x.columns

y = dataset_ready['Risk_bad'].values

# Spliting X and y into train and test version
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=42)

Then, we will experiment with several models to choose a few appropriate ones. Here are the models we shall experiment with:
- [RandomForestClassifier](https://scikit-learn.org/stable/modules/ensemble.html#forests-of-randomized-trees)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression)
- [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier)
- [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier)
- [GaussianNB](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB)
- [Support Vector Machine Classifier (SVC)](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html#sklearn.svm.SVC)
- [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html)
- [XGBoost](https://xgboost.readthedocs.io/en/stable/python/python_api.html#module-xgboost.sklearn)
- More classifiers can be found [here](https://scikit-learn.org/stable/supervised_learning.html#supervised-learning)

In [ ]:
# Experiment with different models 
classifier = LogisticRegression(solver='liblinear')
# classifier = KNeighborsClassifier()
# classifier = DecisionTreeClassifier()
# classifier = GaussianNB()
# classifier = RandomForestClassifier()
# classifier = SVC()
# classifier = MLPClassifier()
# classifier = XGBClassifier()
# classifier = [...]

classifier_name = classifier.__class__.__name__

scoring_type = 'f1'
kfold = KFold(n_splits=5, random_state=42, shuffle=True) # Ensuring all methods are evaluated on the same fold

score = cross_val_score(classifier, X_train, y_train, cv=kfold, scoring=scoring_type)
print(f'Average {scoring_type} performance of the {classifier_name} model = {np.mean(score)}')
    

# Part 6a. Evaluating the performance of models - Predicting the test set

We have seen many models have good accuracy in our training data. **But how good is our models really?**

Let us see how well our model perdicts the unseen test data.




In [ ]:
#Testing the model 
#Predicting using our model

classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

# Check the obtained results
print(f"Accuracy of our model's prediction {accuracy_score(y_test,y_pred)}")

# Part 6b. Evaluating the performance of models - Setting up Baseline Models

Now, let us set up some dummy baselines perform when predicting the test data:
- **Most Frequent:** Compare against a model that always predicts "good" risk
- **Uniform:** Compare against a model that predicts 50 / 50 good and bad risk (toss of a coin) 
- **Stratified:** Compare against a model that predicts 70% good and 30% bad (randomly) 

In [ ]:
from sklearn.dummy import DummyClassifier

strategies = ['most_frequent', 'uniform', 'stratified']

for strategy in strategies:

    dummy_clf = DummyClassifier(strategy=strategy)
    dummy_clf.fit(X_train, y_train)

    #Testing the dummy model 
    y_pred = dummy_clf.predict(X_test)

    # Check the obtained results
    print(f'Performance of a baseline using the {strategy}, Accuracy = {f1_score(y_test,y_pred)} ')

# Part 7. Re-evaluating the performance of models - Exploring other performance metrics

Let us try to evaluate our model using other performance metrics:
- Precision
- Recall
- F1 score (harmonic mean between precision and recall)

In [ ]:
#Testing the model 
#Predicting using our  model
y_pred = classifier.predict(X_test)

# Check the obtained results
print(f"""Performance of our choosen model: 
      \t Accuracy = {accuracy_score(y_test, y_pred)} 
      \t Precision = {precision_score(y_test,y_pred)} 
      \t Recall = {recall_score(y_test, y_pred)} 
      \t F1 = {f1_score(y_test, y_pred)}""")

# print(classification_report(y_test, y_pred))
        
strategies = ['most_frequent', 'uniform', 'stratified']

print(f'\nDUMMY Classifiers (Baseline)')
for strategy in strategies:

    dummy_clf = DummyClassifier(strategy=strategy)
    dummy_clf.fit(X_train, y_train)

    #Testing the dummy model 
    y_pred_dummy = dummy_clf.predict(X_test)

    # Check the obtained results
    print(f"""Performance of {strategy} baseline: 
          Accuracy = {accuracy_score(y_test, y_pred_dummy)} 
          Precision = {precision_score(y_test,y_pred_dummy)} 
          Recall = {recall_score(y_test, y_pred_dummy)} 
          F1 = {f1_score(y_test, y_pred_dummy)}""")

In [ ]:
# Plot the confusion matrix
plot_confusion_matrix(classifier, X_test, y_test)

# Part 8. Fine-tuning of the Model(Optional)

In [ ]:
#Seting the Hyper Parameters
# param_grid = {"penalty": ['none', 'l2', 'l1', 'elasticnet'],
#               "solver":['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'],
#               "max_iter": [100, 300]}

# Créating the classifier
# model = LogisticRegression()

# grid_search = GridSearchCV(model, param_grid=param_grid, cv=5, scoring='f1', verbose=1)
# grid_search.fit(X_train, y_train)

In [ ]:
# print(grid_search.best_score_)
# print(grid_search.best_params_)

In [ ]:
# best_model = LogisticRegression(penalty='none', solver='sag', max_iter=300)

# # training with the best params
# best_model.fit(X_train, y_train)

# # testing
# y_pred = best_model.predict(X_test)

# # Check the obtained results
# print(f1_score(y_test,y_pred))
# print(confusion_matrix(y_test, y_pred))
# print(classification_report(y_test, y_pred))

# Part 8. Explaining the model

## What features are the most important?

We measure the importance of a feature by calculating the increase in the model’s prediction error after **permuting the feature**. 
- A feature is “important” if shuffling its values increases the model error, because in this case the model relied on the feature for the prediction. 
- A feature is “unimportant” if shuffling its values leaves the model error unchanged, because in this case the model ignored the feature for the prediction.



In [ ]:
from sklearn.inspection import permutation_importance

# Using the permutation importance method (test set)
result = permutation_importance(
    classifier, X_test, y_test, n_repeats=10, random_state=42, n_jobs=2
)
sorted_idx = result.importances_mean.argsort()

fig, ax = plt.subplots()
fig.set_size_inches(18.5, 10.5)
ax.boxplot(
    result.importances[sorted_idx].T, vert=False, labels=feature_names
)
ax.set_title("Permutation Importances (test set)")
fig.tight_layout()
plt.show()


In [ ]:
# Using the permutation importance method (test set)
result = permutation_importance(
    classifier, X_train, y_train, n_repeats=10, random_state=42, n_jobs=2
)
sorted_idx = result.importances_mean.argsort()

fig, ax = plt.subplots()
fig.set_size_inches(18.5, 10.5)
ax.boxplot(
    result.importances[sorted_idx].T, vert=False, labels=feature_names
)
ax.set_title("Permutation Importances (train set)")
fig.tight_layout()
plt.show()

### Exploring the effect size of each feature

The next steps use ELI5 library which **does not support** the following classifiers:
- GaussianNB
- MLPClassifier
- XGBoost (dependency conflict version)

In [ ]:
import eli5

eli5.show_weights(classifier, feature_names=feature_names.values, top=100)

### Exploring why the model predicts certain classes

Now that we explored the most important features of the model, we can also try to explain **why** the model predicts certain classes for some records.

In [ ]:
classifier.predict(X_test[0:2])

In [ ]:
eli5.show_prediction(classifier, X_test[0], feature_names=feature_names.values, show_feature_values=True)